## Load packages and initialize GEE

In [1]:
# import packages
import ee
import geemap
import pandas as pd

In [2]:
# authenticate the EE api
ee.Authenticate()

True

In [3]:
# initialize the EE api
ee.Initialize(project='y2y-climate-benefits')

## Define GEE datasets

In [4]:
# define EE datasets
# forest carbon sequestration potential
potential = ee.Image('projects/y2y-climate-benefits/assets/inputs/carbon_accumulation_potential_t_ha_yr')

# vector
y2y = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/y2y")
countries = ee.FeatureCollection("USDOS/LSIB/2017")
us_can = ee.FeatureCollection("projects/y2y-climate-benefits/assets/inputs/us_can_simple")

## Calculate Carbon Sequestration Potential

In [ ]:
# multiply carbon potential by 30 years
potential_30 = potential.multiply(30)

# compute per-pixel area in ha
pixel_area_ha = ee.Image.pixelArea().divide(10000)

In [8]:
## Calculate Carbon Sequestration Potential in Y2Y USA
# Calculate total carbon stocks USA only
usa = countries.filter(ee.Filter.eq('COUNTRY_NA', 'United States'))

sequestration_t = (
    potential_30
    .clipToCollection(usa)
    .multiply(pixel_area_ha)
    .rename(['sequestration_t'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=potential_30.projection().nominalScale(),
        maxPixels=1e20
    )
)

sequestration_area = (
    pixel_area_ha
    .clipToCollection(usa)
    .updateMask(potential_30)
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=y2y.geometry(),
        scale=potential_30.projection().nominalScale(),
        maxPixels=1e20
    )
)

# export to drive
ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection([
        ee.Feature(None, {
            'sequestration_potential': sequestration_t,
            'sequestration_area': sequestration_area
            })]),
    description="sequestration_potential_y2y_usa",
    folder="",
    fileFormat="CSV"
).start()

In [10]:
## Calculate Carbon Sequestration Potential in USA
# Calculate total carbon stocks USA only
usa = countries.filter(ee.Filter.eq('COUNTRY_NA', 'United States'))

sequestration_t = (
    potential_30
    .multiply(pixel_area_ha)
    .rename(['sequestration_t'])
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=usa.geometry(),
        scale=potential_30.projection().nominalScale(),
        maxPixels=1e20
    )
)

sequestration_area = (
    pixel_area_ha
    .updateMask(potential_30)
    .reduceRegion( # calc sum
        reducer=ee.Reducer.sum(),
        geometry=usa.geometry(),
        scale=potential_30.projection().nominalScale(),
        maxPixels=1e20
    )
)

# export to drive
ee.batch.Export.table.toDrive(
    collection=ee.FeatureCollection([
        ee.Feature(None, {
            'sequestration_potential': sequestration_t,
            'sequestration_area': sequestration_area
            })]),
    description="sequestration_potential_usa",
    folder="",
    fileFormat="CSV"
).start()

## Add layers to map

In [9]:
# create a map
m = geemap.Map()

# add layers
m.addLayer(potential_30, {}, 'Carbon Accumulation Potential 30 Years')
m.addLayer(pixel_area_ha.updateMask(potential_30), {}, 'Pixel Area Ha')

# Display the map
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…